In [1]:
from datasets import load_dataset

dataset = load_dataset('smilegate-ai/kor_unsmile')
print(dataset)



c:\Users\User\profanity-cleaner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels'],
        num_rows: 15005
    })
    valid: Dataset({
        features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels'],
        num_rows: 3737
    })
})


In [2]:
import pandas as pd

df = dataset['train'].to_pandas()
df.head()

,문장,여성/가족,남성,성소수자,인종/국적,연령,지역,종교,기타 혐오,악플/욕설,clean,개인지칭,labels
0,일안하는 시간은 쉬고싶어서 그런게 아닐까,0,0,0,0,0,0,0,0,0,1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
1,아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...,0,0,0,0,0,0,1,0,0,0,0,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"
2,루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o doin 진짜 띵...,0,0,0,0,0,0,0,0,0,1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
3,홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...,0,0,0,0,0,0,0,0,0,1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
4,아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...,1,0,0,0,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [3]:
print(df['clean'].value_counts())
print(df['clean'].value_counts(normalize=True) * 100)

clean
0    11266
1     3739
Name: count, dtype: int64
clean
0    75.081639
1    24.918361
Name: proportion, dtype: float64


In [4]:
# clean=1(정상) → label=0,  clean=0(비속어) → label=1
df['label'] = 1 - df['clean']

print(df[['문장', 'clean', 'label']].head())
print(df['label'].value_counts())

                                                  문장  clean  label
0                             일안하는 시간은 쉬고싶어서 그런게 아닐까      1      0
1  아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...      0      1
2  루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o  doin 진짜 띵...      1      0
3  홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...      1      0
4  아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...      0      1
label
1    11266
0     3739
Name: count, dtype: int64


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('beomi/kcbert-base')
print("토크나이저 로드 완료")

토크나이저 로드 완료


In [6]:
test = "안녕하세요 반갑습니다"
result = tokenizer(test)

print("원문:", test)
print("토큰:", tokenizer.tokenize(test))
print("숫자:", result['input_ids'])

원문: 안녕하세요 반갑습니다
토큰: ['안녕', '##하세요', '반', '##갑', '##습니다']
숫자: [2, 19017, 8482, 1483, 4981, 8046, 3]


In [7]:
train_df = dataset['train'].to_pandas()
valid_df = dataset['valid'].to_pandas()

train_df['label'] = 1 - train_df['clean']
valid_df['label'] = 1 - valid_df['clean']

# 문장이랑 label만! (원본 labels 등 다른 컬럼 다 버림)
train_df = train_df[['문장', 'label']]
valid_df = valid_df[['문장', 'label']]

print(train_df.head())
print(train_df['label'].value_counts())

                                                  문장  label
0                             일안하는 시간은 쉬고싶어서 그런게 아닐까      0
1  아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...      1
2  루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o  doin 진짜 띵...      0
3  홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...      0
4  아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...      1
label
1    11266
0     3739
Name: count, dtype: int64


In [8]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

print(train_dataset)

Dataset({
    features: ['문장', 'label'],
    num_rows: 15005
})


In [9]:
#토큰화 및 전제 적용

def tokenize_function(examples):
    return tokenizer(
        examples['문장'],
        padding='max_length',   # 길이를 일정하게 맞춘다
        truncation=True,         # 너무 길면 자른다
        max_length=128           # 최대 128 토큰
    )

#전체에 적용
train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)
print(train_tokenized)

Map: 100%|██████████| 3737/3737 [00:00<00:00, 30883.98 examples/s]

Dataset({
    features: ['문장', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 15005
})


In [10]:
columns_to_return = ['input_ids', 'token_type_ids', 'attention_mask', 'label']
train_tokenized.set_format(type='torch', columns=columns_to_return)
valid_tokenized.set_format(type='torch', columns=columns_to_return)

print(train_tokenized)
print("형식 설정 완료")

Dataset({
    features: ['문장', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 15005
})
형식 설정 완료


In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'beomi/kcbert-base',
    num_labels=2              #정상이면0, 비속어면1 -> 2개
)

print("모델 로드 완료")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7000.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

모델 로드 완료


In [12]:
# 모델 평가지표 정의 확인
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)  #가장 높은확률 클래스 선택
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1}
print("평가지표 정의 완료")

평가지표 정의 완료


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    fp16=True,
)
print("TrainingArguments 설정 완료")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments 설정 완료


In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    compute_metrics=compute_metrics,
)

trainer.train()
print("학습 완료")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.314695,0.306415,0.861386,0.910690
2,0.195021,0.370667,0.872090,0.918402
3,0.087838,0.547988,0.876371,0.918403


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]


학습 완료


In [15]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.087838,0.547988,3,0.876371,0.918403


{'eval_loss': 0.547987699508667, 'eval_accuracy': 0.8763714209258764, 'eval_f1': 0.9184033910279054}


In [17]:
# HF Hub 업로드 전에 huggingface-cli login 또는 notebook_login() 필요
# from huggingface_hub import notebook_login
# notebook_login()

HF_MODEL_ID = "illimax/kcbert-profanity"

model.save_pretrained('./kcbert-profanity')
tokenizer.save_pretrained('./kcbert-profanity')

model.push_to_hub(HF_MODEL_ID)
tokenizer.push_to_hub(HF_MODEL_ID)
print(f"업로드 완료: {HF_MODEL_ID}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]
Processing Files (1 / 1): 100%|██████████|  436MB /  436MB, 12.8MB/s  
New Data Upload: 100%|██████████|  436MB /  436MB, 12.8MB/s  
c:\Users\User\profanity-cleaner\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--illimax--kcbert-profanity. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.c

업로드 완료: illimax/kcbert-profanity
